# Vector stores and semantic search



In [10]:
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np

## Part I: Basic vector store implementation

In [11]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents = []  
        self.embeddings = [] 

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        texts = [doc.text for doc in documents]
        new_embeddings = self.model.encode(texts)        
        if len(self.embeddings) == 0:
            self.embeddings = np.array(new_embeddings)
        else:
            self.embeddings = np.vstack((self.embeddings, new_embeddings))

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        if len(self.embeddings) == 0:
            return []

        query_embedding = self.model.encode(query)

        dot_products = np.dot(self.embeddings, query_embedding)

        norm_query = np.linalg.norm(query_embedding)

        norms_embeddings = np.linalg.norm(self.embeddings, axis=1)    

        norms_embeddings[norms_embeddings == 0] = 1e-10

        norm_query = norm_query if norm_query != 0 else 1e-10

        similarities = dot_products / (norms_embeddings * norm_query)
        
        top_indices = np.argsort(similarities)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            score = float(similarities[idx])
            document = self.documents[idx]
            results.append(SearchResult(score=score, document=document))
            
        return results

In [12]:
#Cargar el dataset 
url_csv = "https://raw.githubusercontent.com/ekohrt/animal-fun-facts-dataset/main/animal-fun-facts-dataset.csv" 

df = pd.read_csv(url_csv)
# Lista donde guardaremos todas las instancias
documents_list = []

for index, row in df.iterrows():
    #extraer el texto principal
    texto_documento = str(row['text'])
    
    #construir el diccionario de metadatos manejando posibles valores nulos (NaN)
    metadatos = {
        "animal_name": str(row['animal_name']) if pd.notna(row['animal_name']) else "",
        "source": str(row['source']) if pd.notna(row['source']) else "",
        "media_link": str(row['media_link']) if pd.notna(row['media_link']) else "",
        "wikipedia_link": str(row['wikipedia_link']) if pd.notna(row['wikipedia_link']) else ""
    }
    
    #crear la instancia de Document y agregarla a la lista
    nuevo_documento = Document(text=texto_documento, metadata=metadatos)
    documents_list.append(nuevo_documento)

In [15]:
# cargar el all-MiniLM-L6-v2
print("Cargando el modelo")
modelo = SentenceTransformer('all-MiniLM-L6-v2')

vector_store = VectorStore(embedding_model=modelo)

print("Generando vectores y agregando documentos a la base de datos")
vector_store.add_documents(documents_list)

print(f"vectorStore contiene {len(vector_store.documents)} documentos y una matriz de embeddings de tamaño {vector_store.embeddings.shape}.")

Cargando el modelo


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6977.41it/s]


Generando vectores y agregando documentos a la base de datos
vectorStore contiene 7734 documentos y una matriz de embeddings de tamaño (7734, 384).


## Consultas

In [40]:
 # Consulta 1
query = "Which animal has a very long neck?"

print("=" * 80)
print(f"CONSULTA: '{query}'")
print("=" * 80)

resultados = vector_store.search(query, top_k=2)

for i, res in enumerate(resultados, 1):
    print(f"  [{i}] Score: {res.score:.4f}")
    print(f"      Texto: {res.document.text}")
    print(f"      Meta:  {res.document.metadata}")
    print(f"      " + "-" * 74)

CONSULTA: 'Which animal has a very long neck?'
  [1] Score: 0.6112
      Texto: Has canines that can be two inches long!
      Meta:  {'animal_name': 'clouded leopard', 'source': 'https://a-z-animals.com/animals/clouded-leopard/', 'media_link': '', 'wikipedia_link': '/wiki/Clouded_leopard'}
      --------------------------------------------------------------------------
  [2] Score: 0.5880
      Texto: Their tails can be up to 5 times the length of their body.
Their strong prehensile tails can be very long, in some cases 5 times the length of their body.
      Meta:  {'animal_name': 'howler monkey', 'source': 'https://factanimal.com/howler-monkey/', 'media_link': '', 'wikipedia_link': '/wiki/Howler_monkey'}
      --------------------------------------------------------------------------


In [29]:
 # Consulta 2
query = "Smart animals that live in the ocean"

print("=" * 80)
print(f"CONSULTA: '{query}'")
print("=" * 80)

resultados = vector_store.search(query, top_k=2)

for i, res in enumerate(resultados, 1):
    print(f"  [{i}] Score: {res.score:.4f}")
    print(f"      Texto: {res.document.text}")
    print(f"      Meta:  {res.document.metadata}")
    print(f"      " + "-" * 74)

CONSULTA: 'Smart animals that live in the ocean'
  [1] Score: 0.6070
      Texto: Smallest cetacean in the ocean
      Meta:  {'animal_name': 'vaquita', 'source': 'https://a-z-animals.com/animals/vaquita/', 'media_link': '', 'wikipedia_link': '/wiki/Vaquita'}
      --------------------------------------------------------------------------
  [2] Score: 0.5871
      Texto: They’re apparently the fastest animal in the ocean.
Early estimates of the speed of sailfish from the ‘40s suggested it could swim at bursts of up to 67mph, making it significantly faster than pretty much everything in the ocean.
      Meta:  {'animal_name': 'sailfish', 'source': 'https://factanimal.com/sailfish/', 'media_link': '', 'wikipedia_link': '/wiki/Sailfish'}
      --------------------------------------------------------------------------


In [28]:
 # Consulta 3
query = "What is the fastest land animal?"
print("=" * 80)
print(f"CONSULTA: '{query}'")
print("=" * 80)

resultados = vector_store.search(query, top_k=2)

for i, res in enumerate(resultados, 1):
    print(f"  [{i}] Score: {res.score:.4f}")
    print(f"      Texto: {res.document.text}")
    print(f"      Meta:  {res.document.metadata}")
    print(f"      " + "-" * 74)

CONSULTA: 'What is the fastest land animal?'
  [1] Score: 0.8592
      Texto: The fastest land mammal in the world!
      Meta:  {'animal_name': 'cheetah', 'source': 'https://a-z-animals.com/animals/cheetah/', 'media_link': '', 'wikipedia_link': '/wiki/Cheetah'}
      --------------------------------------------------------------------------
  [2] Score: 0.8144
      Texto: Fastest animal on Earth
      Meta:  {'animal_name': 'peregrine falcon', 'source': 'https://a-z-animals.com/animals/peregrine-falcon/', 'media_link': '', 'wikipedia_link': '/wiki/Peregrine_falcon'}
      --------------------------------------------------------------------------


In [30]:
#consulta 4
query = "Birds that cannot fly but can swim"
print("=" * 80)
print(f"CONSULTA: '{query}'")
print("=" * 80)

resultados = vector_store.search(query, top_k=2)

for i, res in enumerate(resultados, 1):
    print(f"  [{i}] Score: {res.score:.4f}")
    print(f"      Texto: {res.document.text}")
    print(f"      Meta:  {res.document.metadata}")
    print(f"      " + "-" * 74)

CONSULTA: 'Birds that cannot fly but can swim'
  [1] Score: 0.7261
      Texto: Not all birds are able to fly!
      Meta:  {'animal_name': 'bird', 'source': 'https://a-z-animals.com/animals/bird/', 'media_link': '', 'wikipedia_link': '/wiki/Bird'}
      --------------------------------------------------------------------------
  [2] Score: 0.6969
      Texto: They technically can’t swim.
Despite living in water, their propulsion methods aren’t traditional.
      Meta:  {'animal_name': 'lamprey', 'source': 'https://factanimal.com/lamprey/', 'media_link': '', 'wikipedia_link': '/wiki/Lamprey'}
      --------------------------------------------------------------------------


In [31]:
# Consulta 5
query = "Insects that make honey"
print("=" * 80)
print(f"CONSULTA: '{query}'")
print("=" * 80)

resultados = vector_store.search(query, top_k=2)

for i, res in enumerate(resultados, 1):
    print(f"  [{i}] Score: {res.score:.4f}")
    print(f"      Texto: {res.document.text}")
    print(f"      Meta:  {res.document.metadata}")
    print(f"      " + "-" * 74)

CONSULTA: 'Insects that make honey'
  [1] Score: 0.6419
      Texto: It evolved this relationship with humans long ago.
It is thought that, even before our species evolved, honeyguides were guiding our hominin ancestors to bee nests. 4
      Meta:  {'animal_name': 'honeyguide', 'source': 'https://factanimal.com/honeyguide/', 'media_link': '', 'wikipedia_link': '/wiki/Honeyguide'}
      --------------------------------------------------------------------------
  [2] Score: 0.6218
      Texto: Mainly eats honeybees!
      Meta:  {'animal_name': 'green bee-eater', 'source': 'https://a-z-animals.com/animals/green-bee-eater/', 'media_link': '', 'wikipedia_link': '/wiki/Asian_green_bee-eater'}
      --------------------------------------------------------------------------


## Part II: Filtering by metadata

In [41]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents = []  
        self.embeddings = [] 

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        
        texts = [doc.text for doc in documents]
        new_embeddings = self.model.encode(texts)
        
        if len(self.embeddings) == 0:
            self.embeddings = np.array(new_embeddings)
        else:
            self.embeddings = np.vstack((self.embeddings, new_embeddings))

    def search(self, 
               query: str, 
               top_k: int = 5, 
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        
        if len(self.embeddings) == 0:
            return []

        valid_indices = []
        for i, doc in enumerate(self.documents):
            if metadata_filter is None:
                valid_indices.append(i)
            else:
                coincide = True
                for clave, valor in metadata_filter.items():
                    if doc.metadata.get(clave) != valor:
                        coincide = False
                        break 
                
                if coincide:
                    valid_indices.append(i)
        
        if not valid_indices:
            return []

        query_embedding = self.model.encode(query)
        
        valid_embeddings = self.embeddings[valid_indices]
        
        dot_products = np.dot(valid_embeddings, query_embedding)
        norm_query = np.linalg.norm(query_embedding) if np.linalg.norm(query_embedding) != 0 else 1e-10
        norms_embeddings = np.linalg.norm(valid_embeddings, axis=1)
        norms_embeddings[norms_embeddings == 0] = 1e-10
        
        similarities = dot_products / (norms_embeddings * norm_query)
        
        limite = min(top_k, len(valid_indices))
        top_relative_indices = np.argsort(similarities)[::-1][:limite]
        
        results = []
        for rel_idx in top_relative_indices:
            original_idx = valid_indices[rel_idx]
            score = float(similarities[rel_idx])
            results.append(SearchResult(score=score, document=self.documents[original_idx]))
            
        return results

In [48]:
# 1. Cargar el dataset de noticias AG News
url_ag_news = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/test.csv"

# no tiene encabezados asi que se tienen que asignar
nombres_columnas = ['Class Index', 'Title', 'Description']
df_news = pd.read_csv(url_ag_news, names=nombres_columnas)

# mapear los indices de clase a nombres reales 
categorias = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
df_news['Category'] = df_news['Class Index'].map(categorias)

news_documents = []

for index, row in df_news.iterrows():
    texto_noticia = f"{row['Title']}. {row['Description']}"
    
    metadatos = {
        "category": row['Category']
    }
    
    nuevo_documento = Document(text=texto_noticia, metadata=metadatos)
    news_documents.append(nuevo_documento)

In [49]:
filtered_vector_store = FilteredVectorStore(embedding_model=modelo)

print("Generando vectores y agregando las noticias a la base de datos")
filtered_vector_store.add_documents(news_documents)

print(f"El FilteredVectorStore contiene {len(filtered_vector_store.documents)} documentos")

Generando vectores y agregando las noticias a la base de datos
El FilteredVectorStore contiene 7600 documentos


## Consultas

In [56]:
# Consulta 1: Exploración espacial, pero solo en la sección de Ciencia y Tecnología
query = "missions to space, astronauts and rockets"
filtro = {"category": "Sci/Tech"}

print("=" * 80)
print(f"CONSULTA: '{query}'")
print(f"FILTRO APLICADO: {filtro}")
print("=" * 80)

resultados_1 = filtered_vector_store.search(query, top_k=2, metadata_filter=filtro)

for i, res in enumerate(resultados_1, 1):
    print(f"  [{i}] Score: {res.score:.4f}")
    print(f"      Texto: {res.document.text}")
    print(f"      Meta:  {res.document.metadata}")
    print(f"      " + "-" * 74)

CONSULTA: 'missions to space, astronauts and rockets'
FILTRO APLICADO: {'category': 'Sci/Tech'}
  [1] Score: 0.5232
      Texto: Nasa to resume shuttle missions. The American space agency Nasa says the first space shuttle mission since the Columbia disaster of 2003 is to be launched next May or early June.
      Meta:  {'category': 'Sci/Tech'}
      --------------------------------------------------------------------------
  [2] Score: 0.5136
      Texto: Astronauts arrive at space station. A RUSSIAN spacecraft has delivered three astronauts to the International Space Station, overcoming docking system problems which had delayed its launch.
      Meta:  {'category': 'Sci/Tech'}
      --------------------------------------------------------------------------


In [57]:
# Consulta 2: Acciones y mercado financiero, solo en Negocios
query = "stock market prices and shares dropping"
filtro = {"category": "Business"}

print("=" * 80)
print(f"CONSULTA: '{query}'")
print(f"FILTRO APLICADO: {filtro}")
print("=" * 80)

resultados_1 = filtered_vector_store.search(query, top_k=2, metadata_filter=filtro)

for i, res in enumerate(resultados_1, 1):
    print(f"  [{i}] Score: {res.score:.4f}")
    print(f"      Texto: {res.document.text}")
    print(f"      Meta:  {res.document.metadata}")
    print(f"      " + "-" * 74)

CONSULTA: 'stock market prices and shares dropping'
FILTRO APLICADO: {'category': 'Business'}
  [1] Score: 0.6155
      Texto: Stocks retreat as crude rises. The Dow Jones industrial average fell 38.86, or 0.4 percent, to 10,177.68. The Standard  amp; Poor #39;s 500 index was down 0.69, or 0.1 percent, at 1,134.
      Meta:  {'category': 'Business'}
      --------------------------------------------------------------------------
  [2] Score: 0.6044
      Texto: US Stocks Drop, Led By Drug Shares; Pfizer, Eli Lilly Tumble. US stocks fell as setbacks for drugmakers, including a study showing Pfizer Inc. #39;s Celebrex painkiller increased the risk of heart attacks, sent health-care shares tumbling.
      Meta:  {'category': 'Business'}
      --------------------------------------------------------------------------


In [58]:
# Consulta 3: Partidos, jugadores y campeonatos, solo en Deportes
query = "championship tournament players winning the game"
filtro = {"category": "Sports"}

print("=" * 80)
print(f"CONSULTA: '{query}'")
print(f"FILTRO APLICADO: {filtro}")
print("=" * 80)

resultados_1 = filtered_vector_store.search(query, top_k=2, metadata_filter=filtro)

for i, res in enumerate(resultados_1, 1):
    print(f"  [{i}] Score: {res.score:.4f}")
    print(f"      Texto: {res.document.text}")
    print(f"      Meta:  {res.document.metadata}")
    print(f"      " + "-" * 74)

CONSULTA: 'championship tournament players winning the game'
FILTRO APLICADO: {'category': 'Sports'}
  [1] Score: 0.5340
      Texto: USA searches for winning formula. Hal Sutton anticipated the question. He had formulated an answer, too, long before he arrived in that Milwaukee hotel ballroom to announce his captain #39;s picks and finalize the US Ryder Cup team.
      Meta:  {'category': 'Sports'}
      --------------------------------------------------------------------------
  [2] Score: 0.5247
      Texto: This week #39;s golf tournaments. Last year: Meg Mallon won the season-ending tournament for her lone 2003 title, beating Annika Sorenstam by a stroke. Last week: Heather Daly-Donofrio won the Tournament of Champions in Mobile, Ala.
      Meta:  {'category': 'Sports'}
      --------------------------------------------------------------------------


In [59]:
# Consulta 4: Tratados de paz o conflictos internacionales, solo en Mundo
query = "international peace treaties and global conflicts"
filtro = {"category": "World"}

print("=" * 80)
print(f"CONSULTA: '{query}'")
print(f"FILTRO APLICADO: {filtro}")
print("=" * 80)

resultados_1 = filtered_vector_store.search(query, top_k=2, metadata_filter=filtro)

for i, res in enumerate(resultados_1, 1):
    print(f"  [{i}] Score: {res.score:.4f}")
    print(f"      Texto: {res.document.text}")
    print(f"      Meta:  {res.document.metadata}")
    print(f"      " + "-" * 74)

CONSULTA: 'international peace treaties and global conflicts'
FILTRO APLICADO: {'category': 'World'}
  [1] Score: 0.5333
      Texto: Chirac #39;s tour of China magnifies partnership. Dialogue between China and France, two countries which highly value cultural diversity and pluralism in international politics, is no doubt conducive to world peace.
      Meta:  {'category': 'World'}
      --------------------------------------------------------------------------
  [2] Score: 0.4959
      Texto: Egypt and Israel sign US trade accord 25 years after peace treaty (AFP). AFP - In a partnership hailed as a major boost to often chilly ties, Egypt and Israel signed a first joint trade accord with the United States since their historic peace treaty 25 years ago.
      Meta:  {'category': 'World'}
      --------------------------------------------------------------------------


In [60]:
# Consulta 5: Tecnología, pero filtrado en la sección de Negocios
query = "new software computers and tech companies"
filtro = {"category": "Business"}

print("=" * 80)
print(f"CONSULTA: '{query}'")
print(f"FILTRO APLICADO: {filtro}")
print("=" * 80)

resultados_1 = filtered_vector_store.search(query, top_k=2, metadata_filter=filtro)

for i, res in enumerate(resultados_1, 1):
    print(f"  [{i}] Score: {res.score:.4f}")
    print(f"      Texto: {res.document.text}")
    print(f"      Meta:  {res.document.metadata}")
    print(f"      " + "-" * 74)

CONSULTA: 'new software computers and tech companies'
FILTRO APLICADO: {'category': 'Business'}
  [1] Score: 0.5649
      Texto: Microsoft Opens Software Development Center in India (Update3). Microsoft Corp., the world #39;s largest software maker, will hire several hundred #39; #39; people in the next year at its development center in India, expanding its workforce of 800, Chief Executive Steve Ballmer said.
      Meta:  {'category': 'Business'}
      --------------------------------------------------------------------------
  [2] Score: 0.5538
      Texto: Software giant Microsoft rings up multibillion dollar profit gain. REDMOND, Washington, Oct 21 (AFP) - The world #39;s biggest software company, Microsoft Corp, said Thursday that its first quarter profits swelled to 2.9 billion dollars as consumers and businesses pumped up demand for new computers.
      Meta:  {'category': 'Business'}
      --------------------------------------------------------------------------


# Reflexiones

Como reflexion, la implementación de esto desde cero muestra como la transformación de texto a representaciones matematicas permite realizar busquedas fuera de la simple coincidencia de palabras para lograr una verdadera busqueda semántica. Las diferencias operativas entre ambos enfoques se hicieron evidentes. Por un lado, el VectorStore tradicional ofrece la ventaja de realizar búsquedas amplias y descubrir relaciones conceptuales inesperadas en todo el dataset, pero su principal desventaja es el alto costo computacional y la lentitud, ya que el algoritmo se ve forzado a calcular la similitud del coseno contra cada vector existente. Por otro lado, el FilteredVectorStore soluciona este problema de rendimiento al usar metadatos como un filtro previo, reduce drásticamente el número de operaciones matematicas y garantiza que los resultados pertenezcan exactamente al dominio deseado, aunque su desventaja radica en su estricta dependencia de la calidad de los datos, corriendo el riesgo de omitir información valiosa si los metadatos están incompletos o si el filtro aplicado resulta demasiado restrictivo.